# Problem 2. 가정 일일 전력 사용량 예측하기

## 문제 개요
전력 회사는 가정별 수요를 미리 알아야 발전량을 계획할 수 있습니다.
프랑스 한 가정의 약 4년간 **일일 전력 사용량(kWh)** 기록으로, 그 이후 **90일 동안의 일일 사용량**을 예측하세요.

## 파일 구조
```
Problem2/
├── problem2.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 2006-12-17 ~ 2010-08-28, 1,351일 (date, kwh)
- `data/test.csv` : 2010-08-29 ~ 2010-11-26, 90일 (date만)

| 컬럼 | 설명 |
|---|---|
| date | 날짜 |
| **kwh** | **그날 가정 전체의 유효 전력 사용량(kWh), 예측 대상** |

### 데이터 특징
- 날짜 말고는 **다른 정보가 없는** 순수 시계열 예측 문제입니다.
- 측정기 고장으로 기록이 절반 이상 빠진 날은 kwh가 결측입니다.
- 난방(겨울)·휴가(여름)·요일 등 사람의 생활 패턴이 그대로 드러납니다.
- 출처: Individual Household Electric Power Consumption (UCI Machine Learning Repository, CC BY 4.0)의 분 단위 기록을 일 단위로 집계

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem2/`)에 **`submission.csv`** 로 저장

## 평가
**MAE (평균 절대 오차, kWh)** — 낮을수록 좋습니다.

- 테스트 기간 중 실측값이 없는 4일은 채점에서 제외합니다. 제출 파일에는 90일을 모두 포함하세요.

## 초기 코드 (베이스라인)
아래 베이스라인은 최근 7일 평균이 90일 내내 이어진다고 가정합니다. 동작은 하지만 점수가 낮습니다.

In [1]:
import pandas as pd
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv"); test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "test:", test.shape)

train: (1351, 2) test: (90, 1)


In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

s = train.assign(date=pd.to_datetime(train.date)).set_index("date").kwh          # 일 단위 시계열 (결측 NaN 유지)
test_dates = pd.to_datetime(test.date)
START = s.index[0]
# 2008-08-06~30: 집이 비어 있던 것으로 보이는 이상 구간(4.2~4.6 일정). 학습/프로필에서 제외 (백테스트에서 제외가 유리했음)
ANOM = (pd.Timestamp("2008-08-06"), pd.Timestamp("2008-08-30"))
HOL = {(1, 1), (5, 1), (5, 8), (7, 14), (8, 15), (11, 1), (11, 11), (12, 25)}       # 프랑스 고정 공휴일

def feats(dates, lin=False):
    d = pd.DatetimeIndex(dates); doy = d.dayofyear.values; ang = 2 * np.pi * doy / 365.25
    X = pd.DataFrame({"t": (d - START).days / 365.25, "hol": [int((x.month, x.day) in HOL) for x in d]}, index=d)
    if lin:                                                                      # 선형 모델: 푸리에 3차 + 요일 더미
        for k in (1, 2, 3): X[f"s{k}"] = np.sin(k * ang); X[f"c{k}"] = np.cos(k * ang)
        for k in range(7): X[f"d{k}"] = (d.dayofweek == k).astype(int)
    else:                                                                        # 트리 모델: 요일/월/연중일 + 푸리에 2차
        X["dow"] = d.dayofweek; X["month"] = d.month; X["doy"] = doy
        for k in (1, 2): X[f"s{k}"] = np.sin(k * ang); X[f"c{k}"] = np.cos(k * ang)
    return X

def dow_factors(tr):                                                             # 요일 배수: 값 / 7일 중심 이동평균 의 요일별 평균
    r = tr / tr.rolling(7, center=True, min_periods=5).mean()
    return r.groupby(r.index.dayofweek).mean()

def profile_pred(tr, dates, win=7):
    """과거 연도의 같은 달·일 ±win일 평균 수준의 중앙값 x 요일 배수 (예측 연도 이전 연도만 사용)"""
    L = tr.rolling(2 * win + 1, center=True, min_periods=win).mean(); f = dow_factors(tr); years = sorted(set(tr.index.year)); out = []
    for d in dates:
        vals = []
        for y in years:
            if y >= d.year: continue
            try: dd = pd.Timestamp(year=y, month=d.month, day=d.day)
            except ValueError: dd = pd.Timestamp(year=y, month=d.month, day=28)
            if dd in L.index and not np.isnan(L[dd]): vals.append(L[dd])
        out.append(np.median(vals) * f[d.dayofweek] if vals else np.nan)
    return np.array(out)

def predict_all(history, dates, mask=True):
    tr = history.copy()
    if mask: tr[(tr.index >= ANOM[0]) & (tr.index <= ANOM[1])] = np.nan
    P = {"last7": np.full(len(dates), history.dropna().iloc[-7:].mean())}
    P["prof"] = profile_pred(tr, dates)                       # 수준 보정 없음 (과거 연도 프로필 중앙값 x 요일 배수)
    fit = tr.dropna(); Xtr, Xte = feats(fit.index), feats(dates)
    P["hgb"] = HistGradientBoostingRegressor(loss="absolute_error", max_iter=200, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=15, random_state=0).fit(Xtr, fit.values).predict(Xte)
    P["rf"] = RandomForestRegressor(200, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(Xtr, fit.values).predict(Xte)
    P["lin"] = LinearRegression().fit(feats(fit.index, True), fit.values).predict(feats(dates, True))
    for k in P: P[k] = np.where(np.isnan(P[k]), P["last7"], P[k])
    P["ens_v2"] = (P["prof"] + P["hgb"] + P["rf"] + P["lin"]) / 4                  # 4모델 평균 (선형·HGB 가 9월을 낮게 예측하는 경향)
    P["e7"] = (P["prof"] + P["rf"]) / 2                                            # 프로필 + RF (9월 편향이 작음)
    w = np.clip(1 - np.array([(x - pd.Timestamp(x.year, 8, 31)).days for x in pd.DatetimeIndex(dates)]) / 30.0, 0, 1)   # 8/31 까지 e7, 9/30 에 ens_v2 로 선형 전환
    P["ens"] = w * P["e7"] + (1 - w) * P["ens_v2"]
    return P

In [3]:
# ---- 시간 순서 백테스트: 과거 연도(2008, 2009)의 같은 시기를 기준일 8/28 +-14일에서 90일 예측 (일반 K-Fold 는 시간 누수) ----
rows = []
for mask in (False, True):
    for y in (2008, 2009):
        for sh in (-14, 0, 14):
            o = pd.Timestamp(f"{y}-08-28") + pd.Timedelta(days=sh); dates = pd.date_range(o + pd.Timedelta(days=1), periods=90)
            act = s.reindex(dates); P = predict_all(s[:o], dates, mask)
            ok = (~((dates >= ANOM[0]) & (dates <= ANOM[1]))) & act.notna().values                # 이상 구간은 평가에서 제외
            for k, p in P.items(): rows.append(dict(mask=mask, model=k, mae=np.mean(np.abs(p - act.values)[ok])))
bt = pd.DataFrame(rows).groupby(["model", "mask"]).mae.mean().unstack().round(3).rename(columns={False: "이상구간 학습포함", True: "이상구간 학습제외"})
print("6구간 평균 MAE (낮을수록 좋음):"); print(bt.loc[["last7", "prof", "hgb", "rf", "lin", "ens_v2", "ens"]])

6구간 평균 MAE (낮을수록 좋음):
mask    이상구간 학습포함  이상구간 학습제외
model                       
last7      13.584     13.584
prof        4.615      4.479
hgb         6.668      4.838
rf          5.261      4.660
lin         5.328      4.766
ens_v2      5.070      4.418
ens         4.951      4.395


In [4]:
# ---- 전체 train(~2010-08-28)으로 재학습하여 test 90일 예측 ----
P = predict_all(s, test_dates, mask=True)
pred = np.clip(P["ens"], 4.0, None)                                                # 최소 대기전력 수준(약 4)보다 낮게는 예측하지 않음
submission = pd.DataFrame({"date": test_dates.dt.strftime("%Y-%m-%d"), "kwh": pred})
m = submission.assign(month=test_dates.dt.month.values).groupby("month").kwh.mean().round(2).to_dict()
print("예측 월 평균:", m, "| 과거 3년 같은 월 평균 약 9월 23.5 / 10월 27 / 11월 31")

예측 월 평균: {8: 21.08, 9: 22.87, 10: 26.55, 11: 30.12} | 과거 3년 같은 월 평균 약 9월 23.5 / 10월 27 / 11월 31


In [5]:
submission.to_csv("submission.csv", index=False)
chk = pd.read_csv("submission.csv")
assert list(chk.columns) == ["date", "kwh"] and len(chk) == 90 and chk.kwh.notna().all() and np.isfinite(chk.kwh).all()
assert (chk.date.values == test.date.values).all() and chk.date.iloc[0] == "2010-08-29" and chk.date.iloc[-1] == "2010-11-26"
chk.head()

,date,kwh
0,2010-08-29,23.004416
1,2010-08-30,19.663158
2,2010-08-31,20.561493
3,2010-09-01,20.689735
4,2010-09-02,19.864188


## 제출 형식
`submission.csv`

| date | kwh |
|---|---|
| 2010-08-29 | 21.5 |
| 2010-08-30 | 24.0 |

- 컬럼명 `date`, `kwh`, 날짜 형식 `YYYY-MM-DD`, 90일 모두 포함
- 포맷이 틀리면 0점 처리됩니다.

In [6]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()

(90, 2)


,date,kwh
0,2010-08-29,23.004416
1,2010-08-30,19.663158
2,2010-08-31,20.561493
3,2010-09-01,20.689735
4,2010-09-02,19.864188
